In [ ]:
# =============================================================================
# CÓDIGO COMPLETO: CARGA DE DATOS + OPTUNA
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("PREDICCIÓN DE RENDIMIENTO ACADÉMICO CON OPTUNA")
print("=" * 70)

# =============================================================================
# PARTE 1: CARGA DE DATOS
# =============================================================================

print("\n" + "=" * 60)
print("1. CARGA DE DATOS")
print("=" * 60)

# Intentar cargar archivo CSV
try:
    df = pd.read_csv('../Semana_04/datos_concatenados.csv')
    print(f"✅ Datos cargados: {len(df):,} registros")
    print(f"   Columnas: {df.shape[1]}")
except FileNotFoundError:   
    print("❌ Archivo 'datos_concatenados.csv' no encontrado. Asegúrate de que el archivo exista en la ruta '../Semana_04/'")
    exit(1)    
# Mostrar información básica
print(f"\n📊 Estadísticas básicas:")
print(f"   Estudiantes únicos: {df['ALUMNO_ID'].nunique():,}")
print(f"   Asignaturas únicas: {df['Asignatura'].nunique()}")
print(f"   Carreras: {df['Carrera'].nunique()}")

# =============================================================================
# PARTE 2: ANÁLISIS EXPLORATORIO
# =============================================================================

print("\n" + "=" * 60)
print("2. ANÁLISIS EXPLORATORIO")
print("=" * 60)

# Distribución de aprobados
if 'Aprobado' in df.columns:
    aprobados = df['Aprobado'].value_counts()
    print(f"\n📈 Distribución de aprobados:")
    print(f"   Aprobados (S): {aprobados.get('S', 0):,} ({aprobados.get('S', 0)/len(df)*100:.1f}%)")
    print(f"   No aprobados (N): {aprobados.get('N', 0):,} ({aprobados.get('N', 0)/len(df)*100:.1f}%)")

# Distribución por asignatura
print(f"\n📚 Top 5 asignaturas:")
print(df['Asignatura'].value_counts().head(5))

# Estadísticas de notas
notas = ['Primer.Par', 'Segundo.Par', 'TPLab.', 'Lab.', 'Proy.']
notas_existentes = [n for n in notas if n in df.columns]
if notas_existentes:
    print(f"\n📊 Estadísticas de notas:")
    print(df[notas_existentes].describe().round(2))

# =============================================================================
# PARTE 3: PREPROCESAMIENTO
# =============================================================================

print("\n" + "=" * 60)
print("3. PREPROCESAMIENTO")
print("=" * 60)

# Crear copia
data = df.copy()

# Seleccionar características
features_numericas = ['Primer.Par', 'Segundo.Par', 'TPLab.', 'Lab.', 'Proy.', 'Convocatoria']
features_numericas_existentes = [f for f in features_numericas if f in data.columns]

features_categoricas = ['Cod.Asign', 'Cod.Car.Sec']
features_categoricas_existentes = [f for f in features_categoricas if f in data.columns]

print(f"\n📌 Características numéricas: {features_numericas_existentes}")
print(f"📌 Características categóricas: {features_categoricas_existentes}")

# Codificar variables categóricas
encoders = {}
for col in features_categoricas_existentes:
    if col in data.columns:
        le = LabelEncoder()
        data[f'{col}_Enc'] = le.fit_transform(data[col].astype(str))
        encoders[col] = le
        print(f"   ✅ Codificada: {col} -> {col}_Enc")

# Crear variable objetivo
if 'Aprobado' in data.columns:
    data['Target'] = data['Aprobado'].map({'S': 1, 'N': 0, 'SI': 1, 'NO': 0})
else:
    # Si no existe, crear desde Firma
    threshold = 50 if data['Firma'].max() > 10 else 5
    data['Target'] = (data['Firma'] >= threshold).astype(int)
    print(f"   Target creado desde Firma (threshold={threshold})")

# Características finales
features_final = features_numericas_existentes.copy()
for col in features_categoricas_existentes:
    if f'{col}_Enc' in data.columns:
        features_final.append(f'{col}_Enc')

print(f"\n✅ Features finales: {len(features_final)}")

# Limpiar datos
data_clean = data[features_final + ['Target']].dropna()
print(f"✅ Datos limpios: {len(data_clean):,} registros")

# Separar X y y
X = data_clean[features_final].values
y = data_clean['Target'].values

print(f"\n📊 Distribución final:")
print(f"   Clase 0 (No pasa): {sum(y==0):,} ({sum(y==0)/len(y)*100:.1f}%)")
print(f"   Clase 1 (Pasa): {sum(y==1):,} ({sum(y==1)/len(y)*100:.1f}%)")

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Escalar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ División de datos:")
print(f"   Entrenamiento: {len(X_train):,}")
print(f"   Prueba: {len(X_test):,}")

# Guardar preprocesadores
joblib.dump(scaler, 'scaler.pkl')
for col, le in encoders.items():
    joblib.dump(le, f'encoder_{col}.pkl')
print("✅ Preprocesadores guardados")

# =============================================================================
# PARTE 4: OPTUNA - OPTIMIZACIÓN DE HIPERPARÁMETROS
# =============================================================================

print("\n" + "=" * 60)
print("4. OPTIMIZACIÓN CON OPTUNA")
print("=" * 60)

def create_model(trial):
    """Crea modelo con hiperparámetros sugeridos por Optuna"""
    
    n_layers = trial.suggest_int('n_layers', 1, 4)
    
    model = Sequential()
    
    # Primera capa
    n_units = trial.suggest_int('n_units_0', 16, 256)
    activation = trial.suggest_categorical('activation_0', ['relu', 'tanh'])
    model.add(Dense(n_units, activation=activation, input_shape=(X_train_scaled.shape[1],)))
    model.add(Dropout(trial.suggest_float('dropout_0', 0.0, 0.5)))
    
    # Capas ocultas adicionales
    for i in range(1, n_layers):
        n_units = trial.suggest_int(f'n_units_{i}', 8, 128)
        activation = trial.suggest_categorical(f'activation_{i}', ['relu', 'tanh'])
        model.add(Dense(n_units, activation=activation))
        model.add(Dropout(trial.suggest_float(f'dropout_{i}', 0.0, 0.5)))
    
    # Capa de salida
    model.add(Dense(1, activation='sigmoid'))
    
    # Optimizador y learning rate
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = RMSprop(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

def objective(trial):
    """Función objetivo para Optuna"""
    
    model = create_model(trial)
    
    batch_size = trial.suggest_int('batch_size', 16, 128)
    epochs = trial.suggest_int('epochs', 20, 100)
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        batch_size=batch_size,
        epochs=epochs,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Evaluar
    y_pred_proba = model.predict(X_test_scaled, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    trial.set_user_attr('accuracy', accuracy)
    trial.set_user_attr('precision', precision)
    trial.set_user_attr('recall', recall)
    trial.set_user_attr('auc', auc)
    
    return f1

# Crear estudio
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

print("\n🔍 Optimizando...")
study.optimize(objective, n_trials=30, show_progress_bar=True)

# =============================================================================
# PARTE 5: RESULTADOS
# =============================================================================

print("\n" + "=" * 60)
print("5. RESULTADOS")
print("=" * 60)

best_trial = study.best_trial
print(f"\n✅ Mejor F1-Score: {best_trial.value:.4f}")
print(f"\n📊 Mejores hiperparámetros:")
for key, value in best_trial.params.items():
    print(f"   {key}: {value}")

print(f"\n📈 Métricas del mejor modelo:")
print(f"   Accuracy: {best_trial.user_attrs['accuracy']:.4f}")
print(f"   Precision: {best_trial.user_attrs['precision']:.4f}")
print(f"   Recall: {best_trial.user_attrs['recall']:.4f}")
print(f"   AUC-ROC: {best_trial.user_attrs['auc']:.4f}")

# =============================================================================
# PARTE 6: ENTRENAR MODELO FINAL
# =============================================================================

print("\n" + "=" * 60)
print("6. ENTRENAMIENTO DEL MODELO FINAL")
print("=" * 60)

# Crear modelo con mejores parámetros
best_params = best_trial.params
model_final = create_model(best_trial)

# Entrenar
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model_final.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    batch_size=best_params['batch_size'],
    epochs=best_params['epochs'],
    callbacks=[early_stop],
    verbose=1
)

# Guardar modelo
model_final.save('modelo_rendimiento.h5')
print("✅ Modelo guardado como 'modelo_rendimiento.h5'")

# =============================================================================
# PARTE 7: EVALUACIÓN FINAL
# =============================================================================

print("\n" + "=" * 60)
print("7. EVALUACIÓN FINAL")
print("=" * 60)

# Predicciones
y_pred_proba = model_final.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n📊 Métricas en el conjunto de prueba:")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}")
print(f"   AUC-ROC: {auc:.4f}")

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print(f"\n📊 Matriz de confusión:")
print(f"   VP (Pasa): {cm[1,1]}")
print(f"   VN (No Pasa): {cm[0,0]}")
print(f"   FP: {cm[0,1]}")
print(f"   FN: {cm[1,0]}")

# Reporte de clasificación
print(f"\n📊 Reporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Pasa', 'Pasa']))

# =============================================================================
# PARTE 8: VISUALIZACIONES
# =============================================================================

print("\n" + "=" * 60)
print("8. VISUALIZACIONES")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Evolución del F1-Score
trials_df = study.trials_dataframe()
axes[0, 0].plot(trials_df['number'], trials_df['value'])
axes[0, 0].set_xlabel('Trial')
axes[0, 0].set_ylabel('F1-Score')
axes[0, 0].set_title('Evolución del F1-Score en Optuna')
axes[0, 0].grid(True)

# 2. Importancia de hiperparámetros
importances = optuna.importance.get_param_importances(study)
params = list(importances.keys())
values = list(importances.values())
axes[0, 1].barh(params, values)
axes[0, 1].set_xlabel('Importancia')
axes[0, 1].set_title('Importancia de Hiperparámetros')
axes[0, 1].grid(True)

# 3. Matriz de confusión
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 2])
axes[0, 2].set_xlabel('Predicción')
axes[0, 2].set_ylabel('Real')
axes[0, 2].set_title('Matriz de Confusión')
axes[0, 2].set_xticklabels(['No Pasa', 'Pasa'])
axes[0, 2].set_yticklabels(['No Pasa', 'Pasa'])

# 4. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1, 0].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
axes[1, 0].plot([0, 1], [0, 1], 'k--')
axes[1, 0].set_xlabel('Tasa de Falsos Positivos')
axes[1, 0].set_ylabel('Tasa de Verdaderos Positivos')
axes[1, 0].set_title('Curva ROC')
axes[1, 0].legend()
axes[1, 0].grid(True)

# 5. Distribución de predicciones
axes[1, 1].hist(y_pred_proba[y_test == 0], bins=20, alpha=0.5, label='No Pasa', color='red')
axes[1, 1].hist(y_pred_proba[y_test == 1], bins=20, alpha=0.5, label='Pasa', color='green')
axes[1, 1].axvline(x=0.5, color='black', linestyle='--')
axes[1, 1].set_xlabel('Probabilidad de Pasar')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_title('Distribución de Probabilidades')
axes[1, 1].legend()
axes[1, 1].grid(True)

# 6. Pérdida durante entrenamiento
axes[1, 2].plot(history.history['loss'], label='Entrenamiento')
axes[1, 2].plot(history.history['val_loss'], label='Validación')
axes[1, 2].set_xlabel('Época')
axes[1, 2].set_ylabel('Pérdida')
axes[1, 2].set_title('Evolución de la Pérdida')
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.tight_layout()
plt.savefig('resultados_optuna.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráficos guardados como 'resultados_optuna.png'")

# =============================================================================
# PARTE 9: RESUMEN FINAL
# =============================================================================

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)
print(f"📊 Dataset: {len(df):,} registros, {df.shape[1]} columnas")
print(f"📊 Features: {len(features_final)}")
print(f"📊 Mejor F1-Score: {best_trial.value:.4f}")
print(f"📊 Accuracy final: {accuracy:.4f}")
print(f"📊 AUC-ROC: {auc:.4f}")
print(f"\n📁 Archivos guardados:")
print(f"   - scaler.pkl")
print(f"   - encoder_Cod.Asign.pkl")
print(f"   - encoder_Cod.Car.Sec.pkl")
print(f"   - modelo_rendimiento.h5")
print(f"   - resultados_optuna.png")
print("\n✅ PROCESO COMPLETADO")
print("=" * 70)

IndentationError: expected an indented block after 'except' statement on line 39 (3615800294.py, line 42)